# B2-024 — Practice p18

*65 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

**Data boundary:** use literal unit data and the seeded unit generator (`scripts/generate_capstone_data.py`, read through `data/capstone_data.py`), with the standard library, NumPy, and Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints you did not create in the same run, or student data.

**Set:** C · **Type:** integrative · **Difficulty:** advanced · **Time budget:** 65 minutes  
**Concepts:** semi-supervised-pseudo-labeling

## Task

Certify cluster-then-label against the fixed labelled-only baseline, and compare it with threshold pseudo-labels.

**Data and frozen model.** Use `xl, yl`, `xv, yv`, and `pool` as in p17. Train the labelled-only CNN with exactly the committed recipe: `torch.manual_seed(SEED)`, `model = capstone_data.ShapeCNN()`, `optimizer = torch.optim.Adam(model.parameters(), lr=0.01, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)`, 150 full-batch updates of `F.cross_entropy(model(xl), yl)`. Then freeze it (`model.eval()`, no further updates).

**Frozen features.** `features(x)` returns `model.conv1(x).flatten(1)` computed under `torch.no_grad()`: the first convolution's responses before the ReLU, shape `(N, 288)`.
**Seam:** while a flag `fitting` is true, `features` records `capstone_data.split_of("shapes_ssl", row)` for every input row; set it true only while computing the features that k-means is fitted on. Every recorded value must be `"train"` or `"unlabelled"`.

**k-means (k = 3).** Fit on `Z = torch.cat([features(xl), features(pool)])` (630 rows, labelled first).
Initialize centroid `j` at the mean of `features(xl)[yl == j]` (keep these three as the *labelled-class centroids*).
Run exactly **20** Lloyd iterations: assign each row to the nearest centroid by Euclidean distance (`torch.cdist`, ties to the lower centroid index, as `argmin` does), then set each centroid to the mean of its assigned rows; a centroid with no rows stays unchanged.

**Naming.** Assign each labelled training image to its nearest final centroid. A cluster's name is the majority label of its labelled members, ties to the lowest class index. A cluster with no labelled member takes the label of the nearest labelled-class centroid (Euclidean distance in the frozen feature space).
**Prediction:** an image's class is the name of its nearest final centroid.

**Comparison with threshold pseudo-labels (validation and pool only).** Using the frozen model, select threshold pseudo-labels on the pool with p09's rule, `tau = 0.9` and `max_per_class = 200`. Report (i) the fraction of those selected pool images whose cluster-then-label class agrees with the threshold pseudo-label, and (ii) the validation accuracies of the frozen baseline, of cluster-then-label, and of one round of p17's noisy student trained on `xl` plus these pseudo-labels.

**Evaluation protocol.** Fit only on rows from `capstone_data.train_rows("shapes_ssl")` (and any other rows this statement names), make every choice on `capstone_data.val_rows("shapes_ssl")`, and reach the locked test rows only through `capstone_data.final_test_score(predict_fn, n_boot=1000, alpha=0.05, task="shapes_ssl")`, called exactly once, after all fitting and selection. Assert `capstone_data.test_call_count("shapes_ssl") == 0` immediately before that call and `== 1` after it.
Score cluster-then-label: `predict_fn` returns the cluster names (int64) of the test images.

**Certify.** Test accuracy `>= capstone_data.baseline_score("shapes_ssl").score + 0.03` (bar `0.8400` at the frozen seed); the three cluster names; the seam values all in `{"train", "unlabelled"}`; the test-call protocol.
The `0.03` margin is fixed by the plan; at the frozen seed the reference run clears it with about three points of slack. With other initialization seeds this method sometimes fails to beat the baseline; explain in at most four sentences why the quality of the frozen features decides the outcome, citing your agreement number.

**Required answer-check assertions.** Every certificate item, `Z.shape == (630, 288)`, exactly 20 Lloyd iterations, and the agreement fraction lying in `[0, 1]`.

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

K = 3
LLOYD_ITERATIONS = 20
COMPARISON_THRESHOLD = 0.9
COMPARISON_CAP = 200
MARGIN = 0.03

In [ ]:
# Write the required implementation, probes, certificates, and answer-check assertions here.

## Your response

Show every requested derivation, implementation, probe, certificate, or audit. Use only the permitted data, keep the training, validation, and locked-test roles separate, and end coding work with an answer-check section of executable assertions.